# CMOR example with prototype CMIP6 tables/CVs

This code is a demonstration of the use of CMOR with the MIP tables in this repository and prototype CVs. This requires a conda environment with CMOR v3.15.1 installed (see cmor_environment.yml in this repository)

This example is up to date with the CMIP6 global attributes, including file and directory naming.

In [2]:
%time
import cmor
import numpy 
import json

CPU times: user 3 μs, sys: 0 ns, total: 3 μs
Wall time: 5.96 μs


Identify CMOR version

In [3]:
{i:j for i,j in vars(cmor).items() if 'CMOR_VERSION' in i}

{'CMOR_VERSION_MAJOR': 3, 'CMOR_VERSION_MINOR': 15, 'CMOR_VERSION_PATCH': 1}

The following is adapted from the code at https://github.com/PCMDI/cmor/blob/main/Test/test_cmor_CMIP7.py

Construct the input json file for CMOR


In [4]:
%time

DATASET_INFO = {
    "_AXIS_ENTRY_FILE": "tables/UKCI_coordinate.json",
    "_FORMULA_VAR_FILE": "tables/UKCI_formula_terms.json",
    "_cmip6_option": 1,
    "_controlled_vocabulary_file": "tables/UKCI_CV.json",  # SEPARATE TO MIP TABLES FOR TESTING ONLY
    "activity_id": "CMIP",
    "calendar": "360_day",
    "forcing_index": "1",
    "grid_label": "gm",
    "initialization_index": "1",
    "institution_id": "MOHC",
    "nominal_resolution": "100 km",
    "physics_index": "1",
    "realization_index": "1",
    "source_id": "HadGEM3-GC31-LL",
    "frequency": "mon",
    "mip_era": "UKCI",
    "Conventions": "CF-1.12",
    "grid": "gm",
    "license": "UKCI data produced by MOHC is licensed under the Open Government License v3 (https://www.nationalarchives.gov.uk/doc/open-government-licence/version/3/)",
    "outpath": ".",
    "experiment_id": "1pctCO2",
    "source_type": "AOGCM",
    "parent_activity_id": "none",
    "parent_experiment_id": "none",
    "sub_experiment_id": "none",
    "table_id": "Amon",
    "required_model_components": "AOGCM",
    "branch_time_in_child": "0.0D0",
    "branch_time_in_parent": "0.0D0",
    "parent_time_units": "days since 1850-01-01 00:00:00",
    "parent_variant_label": "r1i1p1f1",
    "parent_source_id": "HadGEM3-GC31-LL",
    "parent_mip_era": "CMIP6",
    "branch_method": "no parent",
    "sub_experiment": "none"
}

with open('input.json', 'w') as fh:
    json.dump(DATASET_INFO, fh, indent=2)

CPU times: user 2 μs, sys: 0 ns, total: 2 μs
Wall time: 5.25 μs


In [5]:
%time
cmor.setup(inpath="tables", netcdf_file_action=cmor.CMOR_REPLACE)

cmor.dataset_json('input.json')

tos = numpy.array([27.] * 24)
tos = tos.reshape(2, 3, 4)

lat = numpy.array([10., 20., 30.])
lat_bnds = numpy.array([5., 15., 25., 35.])
lon = numpy.array([0., 90., 180., 270.])
lon_bnds = numpy.array([-45., 45.,
                        135.,
                        225.,
                        315.
                        ])
time = numpy.array([15.0, 45.0])
time_bnds = numpy.array([0.0, 30.0, 60.0])

CPU times: user 2 μs, sys: 1 μs, total: 3 μs
Wall time: 4.77 μs


In [6]:
%time
realm = "Amon"
cmor.load_table(f"UKCI_{realm}.json")

CPU times: user 2 μs, sys: 1 μs, total: 3 μs
Wall time: 5.25 μs


0

In [7]:
%time
cmorlat = cmor.axis("latitude",
                    coord_vals=lat,
                    cell_bounds=lat_bnds,
                    units="degrees_north")
cmorlon = cmor.axis("longitude",
                    coord_vals=lon,
                    cell_bounds=lon_bnds,
                    units="degrees_east")
cmortime = cmor.axis("time",
                    coord_vals=time,
                    cell_bounds=time_bnds,
                    units="days since 2018")
axes = [cmortime, cmorlat, cmorlon]

CPU times: user 3 μs, sys: 1 μs, total: 4 μs
Wall time: 5.25 μs


In [8]:
variable = "hfss"
cmortos = cmor.variable(variable, "W m-2", axes, positive = "up")


In [9]:
cmor.write(cmortos, tos)
filename = cmor.close(cmortos, file_name=True)
filename

'./UKCI/CMIP/MOHC/HadGEM3-GC31-LL/1pctCO2/r1i1p1f1/Amon/hfss/gm/v20260916/hfss_Amon_HadGEM3-GC31-LL_1pctCO2_r1i1p1f1_gm_201801-201802.nc'

Locate generated file

In [10]:
!find UKCI -type f -name '*.nc'

5615.86s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


UKCI/CMIP/MOHC/HadGEM3-GC31-LL/1pctCO2/r1i1p1f1/Amon/hfss/gm/v20260915/hfss_Amon_HadGEM3-GC31-LL_1pctCO2_r1i1p1f1_gm_201801-201802.nc
UKCI/CMIP/MOHC/HadGEM3-GC31-LL/1pctCO2/r1i1p1f1/Amon/hfss/gm/v20260916/hfss_Amon_HadGEM3-GC31-LL_1pctCO2_r1i1p1f1_gm_201801-201802.nc


Dump header of generated file

In [ ]:
!ncdump -h UKCI/CMIP/MOHC/HadGEM3-GC31-LL/1pctCO2/r1i1p1f1/Amon/hfss/gm/v20260915/hfss_Amon_HadGEM3-GC31-LL_1pctCO2_r1i1p1f1_gm_201801-201802.nc

In [ ]:
# clean up
! rm -rf UKCI
! rm -rf input.json